# 🔬 Notebook 2: Standalone Acoustic Calibration Lab ($k(f)$ Extraction)

Welcome to the **Acoustic Calibration Protocol Suite**. This notebook guides you step-by-step through measuring the physical acoustic constant $k(f)$ across multiple frequencies and distances, validating the inverse-distance law $V_{\text{RMS}} = \frac{k(f)}{r}$ ($R^2 \ge 0.95$), and exporting a portable JSON profile for real-time metric distance tracking.

---

### 🏛️ The Physical Acoustic Principle
Acoustic spherical wave decay dictates:
$$V_{\text{RMS}}(r_i,\, f_j) = k(f_j) \cdot \left(\frac{1}{r_i}\right) + c_{\text{room}}(f_j)$$

* **Slope $k(f_j)$:** The true physical speaker-to-microphone acoustic constant in $\text{Volts}\cdot\text{meters}$.
* **Intercept $c_{\text{room}}(f_j)$:** Absorbs constant ambient room reverberation.
* **Linearity Gate $R^2(f) \ge 0.95$:** Validates that measurements were taken within the line-of-sight Direct Field Zone.

## 1. Initialize Hardware Overlay & Calibration Protocol
Instantiate `MicrophoneArrayOverlay` and `AcousticCalibrationProtocol`.

In [ ]:
import time
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

from pynq_localizer import (
    MicrophoneArrayOverlay,
    KinematicAnalytics,
    AcousticCalibrationProtocol,
    AcousticProfile,
    DistanceEstimator
)

# Initialize hardware
ol = MicrophoneArrayOverlay()
protocol = AcousticCalibrationProtocol(r2_threshold=0.95)

print(f"✅ Hardware Overlay Active: {ol.current_profile} mode ({ol.fs_per_ch:.0f} SPS)")
print(f"✅ Calibration Protocol Initialized (R² Quality Gate >= {protocol.r2_threshold})")

## 2. Define Calibration Grid (Distances & Frequencies)
Configure the physical measurement grid. Standard recommendations:
* **Distances:** 3 or 4 points within the direct-field linear zone (e.g. $0.30\,\text{m}, 0.50\,\text{m}, 0.80\,\text{m}$).
* **Frequencies:** Key acoustic carrier tones (e.g. $1000\,\text{Hz}, 1500\,\text{Hz}, 2000\,\text{Hz}, 2500\,\text{Hz}$).

In [ ]:
# Calibration Grid Configuration
calib_distances_m = [0.30, 0.50, 0.80]             # Tape-measured distances in meters
calib_frequencies_hz = [1000.0, 1500.0, 2000.0, 2500.0] # Carrier frequencies to calibrate in Hz

print("Calibration Grid Configured:")
print(f"  • Distances   : {calib_distances_m} meters")
print(f"  • Frequencies : {calib_frequencies_hz} Hz")

## 3. Guided Interactive Sweep Capture
Run this cell when you are ready with your phone tone generator. For each distance, place the phone at the tape-measured distance, play the requested frequency, and call `capture_calibration_point()`.

In [ ]:
# Interactive Guided Calibration Capture Function
def capture_calibration_point(distance_m, target_freq_hz, num_averages=15):
    """Captures num_averages frames and computes the mean in-band RMS voltage."""
    amps = []
    for _ in range(num_averages):
        frame = ol.capture_quadruple(source="A0", f_min=target_freq_hz-60.0, f_max=target_freq_hz+60.0, timeout=0.35)
        q = frame["quadruple"]
        if q["amplitude_v"] > 0.001:
            amps.append(q["amplitude_v"])
        time.sleep(0.01)
    
    if len(amps) == 0:
        raise ValueError(f"No signal detected for {target_freq_hz} Hz at {distance_m} m. Ensure tone is playing!")
    
    mean_v = float(np.mean(amps))
    protocol.add_measurement(distance_m=distance_m, frequency_hz=target_freq_hz, amplitude_v=mean_v)
    print(f"  ✅ Captured r={distance_m:.2f} m | f={target_freq_hz:.0f} Hz | Mean V_RMS={mean_v*1000:.2f} mV")
    return mean_v

print("Ready for guided calibration. (Call capture_calibration_point for each measurement).")

## 4. Automated 1/r Regression Fitting & Interactive Diagnostics
Fit $1/r$ lines across all frequencies and display interactive Plotly regression curves, $k(f)$ spectra, and $R^2(f)$ quality bars.

In [ ]:
# Fit 1/r Regressions
fit_results = protocol.fit()

# -----------------------------------------------------------------------------
# Plotly Interactive Diagnostic Dashboard
# -----------------------------------------------------------------------------
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "<b>1. V_RMS vs. 1/r Linear Regression Lines</b>",
        "<b>2. Calibrated k(f) Function [V·m]</b>",
        "<b>3. Goodness-of-Fit R²(f) Quality Gate</b>"
    ),
    horizontal_spacing=0.08
)

colors = ["#00FFCC", "#FFA500", "#00E5FF", "#FF007F", "#76FF03"]

freqs_list = []
k_list = []
r2_list = []

for idx, (f, res) in enumerate(fit_results.items()):
    freqs_list.append(f)
    k_list.append(res["k"])
    r2_list.append(res["r_squared"])
    
    # Regression line in Col 1
    points = protocol._raw_measurements[f]
    r_meas = np.array([p[0] for p in points])
    v_meas = np.array([p[1] for p in points]) * 1000.0 # in mV
    
    inv_r = 1.0 / r_meas
    inv_r_line = np.linspace(min(inv_r)*0.9, max(inv_r)*1.1, 50)
    v_line = (res["k"] * inv_r_line + res["c_room"]) * 1000.0
    
    c = colors[idx % len(colors)]
    fig.add_scatter(x=inv_r, y=v_meas, mode="markers", marker=dict(size=8, color=c), name=f"{f:.0f} Hz (Data)", row=1, col=1)
    fig.add_scatter(x=inv_r_line, y=v_line, mode="lines", line=dict(color=c, dash="dash"), name=f"{f:.0f} Hz (Fit)", row=1, col=1)

# Column 2: k(f) continuous curve
fig.add_scatter(x=freqs_list, y=k_list, mode="lines+markers", line=dict(color="#00FFCC", width=2), marker=dict(size=8), name="k(f)", row=1, col=2)

# Column 3: R^2 Bar Chart with 0.95 Quality Gate Line
fig.add_bar(x=[f"{f:.0f} Hz" for f in freqs_list], y=r2_list, marker=dict(color="#00E5FF"), name="R² Score", row=1, col=3)
fig.add_hline(y=0.95, line=dict(color="orange", dash="dash"), annotation_text="R² Gate (0.95)", row=1, col=3)

fig.update_layout(template="plotly_dark", height=450, title="<b>Acoustic Calibration Laboratory: 1/r Regression & Quality Analysis</b>")
fig.update_xaxes(title="1 / Distance [m⁻¹]", row=1, col=1)
fig.update_yaxes(title="V_RMS [mV]", row=1, col=1)
fig.update_xaxes(title="Frequency [Hz]", row=1, col=2)
fig.update_yaxes(title="k [V·m]", row=1, col=2)
fig.update_yaxes(title="R² Score", range=[0.8, 1.01], row=1, col=3)

fig.show()

## 5. Export Calibrated Profile to JSON
Save the validated profile to a portable JSON file.

In [ ]:
export_path = "my_phone_calibrated_profile.json"
saved_file = protocol.save_profile_json(
    filepath=export_path,
    name="Smartphone_Lab_Profile",
    description="Multi-frequency 1/r calibrated profile for smartphone speaker in lab environment"
)

print(f"🎉 SUCCESS: Saved calibrated profile to: {saved_file.resolve()}")

## 6. Live Distance Inversion Verification
Load the newly created profile into `DistanceEstimator` and track distance live in centimeters.

In [ ]:
# Load newly generated profile into DistanceEstimator
loaded_profile = AcousticProfile.from_json(export_path)
estimator = DistanceEstimator(profile=loaded_profile, noise_gate_v=0.003)

# Capture live frame and test metric distance inversion
frame = ol.capture_quadruple(source="A0", f_min=100.0, f_max=15000.0, timeout=0.5)
result = estimator.process_frame(frame, source="A0")

print("\n--- Live Inverted Distance Result ---")
print(f"Detected Pitch f0  : {result['frequency_hz']:.1f} Hz")
print(f"In-Band Amplitude  : {result['amplitude_v']*1000:.2f} mV")
print(f"Evaluated k(f0)    : {result['k_evaluated']:.4f} V·m")
print(f"Inverted Distance  : {result['distance_m']*100:.1f} cm (±{result['distance_err_m']*100:.1f} cm)")

ol.close()
print("\n🔒 Hardware resources cleanly released.")